# Mini RAG System — Movie Plots (LangChain)

A Retrieval-Augmented Generation pipeline over a subset of the [Wikipedia Movie Plots dataset](https://www.kaggle.com/datasets/jrobischon/wikipedia-movie-plots), built with LangChain.

Features like HyDE, self-query filtering, and routing.

Run cells top to bottom. See `README.md` for setup details.


In [ ]:
%pip install -q langchain langchain-core langchain-text-splitters langchain-community langchain-google-genai faiss-cpu rank_bm25 pandas pydantic

In [ ]:
import os
import json
from collections import defaultdict
from typing import Dict, List, Optional

import pandas as pd
from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import DataFrameLoader
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI

## 1. Config

Global configuration and feature flags.


In [4]:
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

# --- LLM ---
GEN_MODEL = "gemini-2.5-flash"

# --- Embeddings ---
EMBED_MODEL = "gemini-embedding-001"

# --- Data ---
DATA_PATH = os.environ.get("RAG_DATA_PATH", "data/wiki_movie_plots_deduped.csv")  # full Kaggle CSV
SAMPLE_SIZE = 300
CHUNK_SIZE = 300       # words per chunk
CHUNK_OVERLAP = 50

# --- Retrieval ---
RETRIEVE_K = 15            # candidates pulled from EACH of the dense/sparse legs before fusion
TOP_K = 5                  # final chunks sent to the LLM
RRF_K = 60                 # standard Reciprocal Rank Fusion damping constant

# --- Feature flags ---
ENABLE_HYDE = True
ENABLE_SELF_QUERY = True
ENABLE_ANSWERABILITY_ROUTING = True


# Chat model via Vertex AI
chat_model = ChatGoogleGenerativeAI(
    model=GEN_MODEL,
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    temperature=0,
)

# Asymmetric embedding clients for docs vs queries
doc_embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBED_MODEL,
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    task_type="RETRIEVAL_DOCUMENT",
)
query_embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBED_MODEL,
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    task_type="RETRIEVAL_QUERY",
)

## 2. Load & chunk the data

Loads movie data into LangChain `Document` objects and chunks text by word count on natural sentence/paragraph boundaries.


In [5]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["Title", "Plot"]).reset_index(drop=True)
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

cast_prefix = df["Cast"].fillna("").apply(lambda c: f"Cast: {c}\n" if c else "")
df["searchable_text"] = "Title: " + df["Title"] + "\n" + cast_prefix + "Plot: " + df["Plot"]

loader = DataFrameLoader(df, page_content_column="searchable_text")
docs = loader.load()

print(f"Loaded {len(docs)} movies from '{DATA_PATH}'")
print(docs[0].metadata)
print(docs[0].page_content[:200], "...")


Loaded 300 movies from 'data/wiki_movie_plots_deduped.csv'
{'Release Year': 1951, 'Title': 'The Day the Earth Stood Still', 'Origin/Ethnicity': 'American', 'Director': 'Robert Wise', 'Cast': 'Michael Rennie, Patricia Neal', 'Genre': 'science fiction', 'Wiki Page': 'https://en.wikipedia.org/wiki/The_Day_the_Earth_Stood_Still_(1951_film)', 'Plot': 'When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges from the saucer and quickly disintegrates the soldiers\' weapons. The alien orders the robot, Gort, to stop. He explains that the now-broken device was a gift for the President which would have enabled him "to study life on the other planets".\r\nThe alien, Klaatu, is taken to Walter Reed Hospital. After surgery, he uses a salve to quickly heal his wound. Meanwhile, the Army is unable to enter the

In [6]:
def word_count(text: str) -> int:
    return len(text.split())


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=word_count,
    add_start_index=True,
)

all_splits = text_splitter.split_documents(docs)
for i, d in enumerate(all_splits):
    d.metadata["chunk_id"] = i  # stable id used later to de-dupe across the dense/sparse steps

print(f"{len(docs)} movies -> {len(all_splits)} chunks")
all_splits[0].metadata


300 movies -> 641 chunks


{'Release Year': 1951,
 'Title': 'The Day the Earth Stood Still',
 'Origin/Ethnicity': 'American',
 'Director': 'Robert Wise',
 'Cast': 'Michael Rennie, Patricia Neal',
 'Genre': 'science fiction',
 'Wiki Page': 'https://en.wikipedia.org/wiki/The_Day_the_Earth_Stood_Still_(1951_film)',
 'Plot': 'When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges from the saucer and quickly disintegrates the soldiers\' weapons. The alien orders the robot, Gort, to stop. He explains that the now-broken device was a gift for the President which would have enabled him "to study life on the other planets".\r\nThe alien, Klaatu, is taken to Walter Reed Hospital. After surgery, he uses a salve to quickly heal his wound. Meanwhile, the Army is unable to enter the saucer; Gort stands outside, silent and unmoving.\r

## 3. Index: dense (FAISS) + sparse (BM25)

Builds an in-memory FAISS vector index alongside a BM25 keyword retriever for hybrid search.


In [7]:
print("Embedding chunks with Google Gemini (Vertex AI)...")
vector_store = FAISS.from_documents(all_splits, doc_embeddings)
print(f"FAISS index built with {vector_store.index.ntotal} vectors")

bm25_retriever = BM25Retriever.from_documents(all_splits)
bm25_retriever.k = RETRIEVE_K
print("BM25 keyword index built")

Embedding chunks with Google Gemini (Vertex AI)...


FAISS index built with 641 vectors
BM25 keyword index built


## 4. Query analysis: answerability routing & self-query filters

Uses a single structured LLM call to classify query in-scope status and extract metadata filters (genre/release year).


In [8]:
YEAR_MIN, YEAR_MAX = int(df["Release Year"].min()), int(df["Release Year"].max())


class QueryAnalysis(BaseModel):
    in_scope: bool = Field(
        description="True if this looks like a question about a movie's plot, characters, or "
        "events that could plausibly be answered from a corpus of movie plot summaries. False "
        "for chit-chat, math, unrelated topics, or opinion/recommendation requests with no plot "
        "content to retrieve."
    )
    reason: str = Field(description="One short sentence justifying the in_scope decision.")
    genre: Optional[str] = Field(
        default=None,
        description="A genre explicitly mentioned in the question (e.g. 'horror', 'science "
        "fiction'). Null if none is mentioned.",
    )
    year_min: Optional[int] = Field(
        default=None,
        description="Earliest release year implied by the question (e.g. 'movies from the 90s' "
        "-> 1990). Null if no year constraint.",
    )
    year_max: Optional[int] = Field(
        default=None, description="Latest release year implied by the question. Null if none."
    )


query_analyzer = chat_model.with_structured_output(QueryAnalysis)

QUERY_ANALYSIS_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You analyze a question against a movie-plot Q&A corpus (release years "
            "{year_min}-{year_max}). Decide whether it's answerable from movie plots, and "
            "extract any genre/year filters implied by the question. Only extract a genre if "
            "the user named a real, unambiguous genre.",
        ),
        ("human", "Question: {question}"),
    ]
)


def analyze_query(question: str, query_analyzer_model) -> QueryAnalysis:
    messages = QUERY_ANALYSIS_PROMPT.format_messages(
        question=question, year_min=YEAR_MIN, year_max=YEAR_MAX
    )
    return query_analyzer_model.invoke(messages)

## 5. Query transformation: HyDE 

Generates a hypothetical plot passage for dense similarity search.


In [9]:
HYDE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write a short, plausible-sounding 2-3 sentence movie plot excerpt that WOULD "
            "answer the question below, as if it were pulled straight from a Wikipedia plot "
            "summary. Invent plausible specifics (character names, actions) -- don't hedge or "
            "say you don't know. This text is only used to improve semantic search and is "
            "never shown to the user.",
        ),
        ("human", "Question: {question}"),
    ]
)


def hyde_passage(question: str, chat_model) -> str:
    return chat_model.invoke(HYDE_PROMPT.format_messages(question=question)).content


## 6. Hybrid retrieval with Reciprocal Rank Fusion

Combines dense (FAISS) and sparse (BM25) search results using RRF scoring, applying metadata filters when present.


In [10]:
def matches_filters(meta: dict, genre: Optional[str], year_min: Optional[int], year_max: Optional[int]) -> bool:
    if genre and genre.lower() not in str(meta.get("Genre", "")).lower():
        return False
    year = meta.get("Release Year")
    if year_min and (year is None or year < year_min):
        return False
    if year_max and (year is None or year > year_max):
        return False
    return True


def reciprocal_rank_fusion(ranked_lists: List[List[Document]], k: int = RRF_K) -> List[Document]:
    """Merge several ranked doc lists into one, scoring each doc by 1/(k + rank) per list it's in."""
    scores: Dict[int, float] = defaultdict(float)
    by_id: Dict[int, Document] = {}
    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked):
            cid = doc.metadata["chunk_id"]
            scores[cid] += 1.0 / (k + rank + 1)
            by_id[cid] = doc
    fused_ids = sorted(scores, key=scores.get, reverse=True)
    return [by_id[cid] for cid in fused_ids]


def hybrid_retrieve(
    question: str,
    vector_store,
    bm25_retriever,
    genre: Optional[str] = None,
    year_min: Optional[int] = None,
    year_max: Optional[int] = None,
    hyde_text: Optional[str] = None,
    top_k: int = TOP_K,
) -> List[Document]:
    # Embed the query (or HyDE passage) with the query-optimized embeddings
    query_vector = query_embeddings.embed_query(hyde_text or question)
    dense_hits = vector_store.similarity_search_by_vector(query_vector, k=RETRIEVE_K)
    sparse_hits = bm25_retriever.invoke(question)

    if genre or year_min or year_max:
        dense_hits = [d for d in dense_hits if matches_filters(d.metadata, genre, year_min, year_max)]
        sparse_hits = [d for d in sparse_hits if matches_filters(d.metadata, genre, year_min, year_max)]
        if not dense_hits and not sparse_hits:
            return []  # Filters were too strict, return empty rather than silently ignoring user constraints

    fused = reciprocal_rank_fusion([dense_hits, sparse_hits])
    return fused[:top_k]

## 7. Structured, grounded generation

Uses `with_structured_output` with `RAGAnswer` (Pydantic) to return validated JSON, grounding contexts in retrieved documents.


In [11]:
class RAGAnswer(BaseModel):
    answer: str = Field(description="A 2-4 sentence natural-language answer to the question.")
    contexts: List[str] = Field(description="The retrieved excerpt(s) used to form the answer.")
    reasoning: str = Field(description="1-2 sentences on which excerpt(s) were used and why.")


SYSTEM_PROMPT = (
    "You are a movie-plot question answering assistant. Answer the user's question using ONLY "
    "the retrieved plot excerpts given as context -- do not use outside knowledge. If the "
    "context doesn't contain enough information to answer confidently, say so plainly in the "
    "answer field instead of guessing."
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "Question: {question}\n\nRetrieved context:\n{context}"),
    ]
)

structured_llm = chat_model.with_structured_output(RAGAnswer)


def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(
        f"[{i + 1}] {d.metadata.get('Title')} ({d.metadata.get('Release Year', '')}): {d.page_content}"
        for i, d in enumerate(docs)
    )


## 8. The full pipeline

Combines routing, query analysis, retrieval, and generation into the main entrypoint `answer_question`.


In [12]:
def answer_question(
    query: str,
    vector_store,
    bm25_retriever,
    chat_model,
    structured_llm,
    prompt,
    query_analyzer_model,
    verbose: bool = True
) -> dict:
    analysis = None
    if ENABLE_ANSWERABILITY_ROUTING or ENABLE_SELF_QUERY:
        analysis = analyze_query(query, query_analyzer_model)

    if ENABLE_ANSWERABILITY_ROUTING and analysis is not None and not analysis.in_scope:
        output = {
            "answer": f"That doesn't look answerable from this movie-plot corpus: {analysis.reason}",
            "contexts": [],
            "reasoning": "Answerability routing short-circuited before retrieval -- no plot chunks were fetched.",
        }
        if verbose:
            print(json.dumps(output, indent=2))
        return output

    genre = analysis.genre if (ENABLE_SELF_QUERY and analysis) else None
    year_min = analysis.year_min if (ENABLE_SELF_QUERY and analysis) else None
    year_max = analysis.year_max if (ENABLE_SELF_QUERY and analysis) else None

    hyde_text = hyde_passage(query, chat_model) if ENABLE_HYDE else None
    top_docs = hybrid_retrieve(
        query,
        vector_store=vector_store,
        bm25_retriever=bm25_retriever,
        genre=genre,
        year_min=year_min,
        year_max=year_max,
        hyde_text=hyde_text,
        top_k=TOP_K
    )

    if not top_docs:
        output = {
            "answer": "I couldn't find any movies matching your specific criteria.",
            "contexts": [],
            "reasoning": "The query filters yielded 0 matching plot chunks from the retrieval step.",
        }
        if verbose:
            print(json.dumps(output, indent=2))
        return output

    context = format_docs(top_docs)
    parsed: RAGAnswer = structured_llm.invoke(prompt.format_messages(question=query, context=context))

    output = parsed.model_dump()
    output["contexts"] = [d.page_content for d in top_docs]  # ground contexts in what was actually retrieved
    if verbose:
        print(json.dumps(output, indent=2))
    return output


## 9. Demo queries

Four queries, each exercising a different part of the pipeline.

**Semantic retrieval** — no exact keyword overlap with the answer ("AI" vs. "HAL 9000"-style plots).

In [ ]:
result_1 = answer_question(
    "Which movie features an artificial intelligence system that turns against the crew?",
    vector_store=vector_store,
    bm25_retriever=bm25_retriever,
    chat_model=chat_model,
    structured_llm=structured_llm,
    prompt=prompt,
    query_analyzer_model=query_analyzer
)


**Hybrid search for actor/plot match** — BM25 + dense search matching actor names and plot details.


In [14]:
result_2 = answer_question(
    "Which movie starring Denzel Washington features a forensics expert?",
    vector_store=vector_store,
    bm25_retriever=bm25_retriever,
    chat_model=chat_model,
    structured_llm=structured_llm,
    prompt=prompt,
    query_analyzer_model=query_analyzer
)


{
  "answer": "The movie \"The Bone Collector\" starring Denzel Washington features a forensics expert. Denzel Washington plays Lincoln Rhyme, a tetraplegic forensics expert who teams up with a patrol cop to solve a string of murders.",
  "contexts": [
    "Title: The Bone Collector\nCast: Denzel Washington, Angelina Jolie\nPlot: The film begins in late 1999. Tetraplegic forensics expert Lincoln Rhyme and a patrol cop, Amelia Donaghy, team up to solve a string of murders connected to a serial killer by his signature: a single shard of bone removed from each of the victims. Rhyme was paralyzed from the neck down in an earlier accident and is bed-bound and completely reliant on machines and his nurse Thelma.\r\nThe killer poses as a New York City taxi driver and abducts and kills those who get in his taxi. The first victims are a married couple named Alan and Lindsay Rubin that the killer picked up at the airport. Amelia finds Alan's body buried in a Civil War-era railroad bed. She also 

**Self-query filters** — exercises the genre/year extraction from Section 4.

In [15]:
result_3 = answer_question(
    "Any horror movies from the 1950s?",
    vector_store=vector_store,
    bm25_retriever=bm25_retriever,
    chat_model=chat_model,
    structured_llm=structured_llm,
    prompt=prompt,
    query_analyzer_model=query_analyzer
)


{
  "answer": "Yes, \"The Alligator People\" is a horror movie from the 1950s, released in 1959.",
  "contexts": [
    "That night, Paul encounters Joyce at the clinic and turns away from her in shame. After seeing Joyce clasping her son's hands and reassuring him of her love, Lavinia apologizes to her for her brusqueness. As Paul climbs onto the table and Mark aims the ray at him, Manon bursts into the lab and destroys the control panel, shooting powerful rays at Paul that transform him into a bipedal, reptilian monster with an alligator-like head. After trying to attack Manon, Paul looks on as Manon's hook is caught on some cords and is electrocuted to death while trying to attack Paul. Confused, Paul stumbles over to the other room and tries to communicate, but his voice has been replaced with a reptilian snarl. Hearing his wife and mother scream in horror, Paul flees into the swamps and sadly peering into the water, sees his reflection. Joyce scrambles after him, as the cobalt mach

**Answerability routing** — out-of-corpus question; should short-circuit before retrieval.

In [ ]:
result_4 = answer_question(
    "What's the weather like today?",
    vector_store=vector_store,
    bm25_retriever=bm25_retriever,
    chat_model=chat_model,
    structured_llm=structured_llm,
    prompt=prompt,
    query_analyzer_model=query_analyzer
)


## Summary of Architecture

- **LLM & Embeddings:** Google Gemini (`gemini-2.5-flash` and `gemini-embedding-001`).
- **Chunking:** `RecursiveCharacterTextSplitter` configured for 300-word chunks.
- **Hybrid Search:** In-memory FAISS dense vector search combined with BM25 sparse keyword search via RRF.
- **Query Analysis:** Extracts genre/year filters and handles answerability routing.
- **Structured Output:** Strictly enforced via Pydantic model (`RAGAnswer`).
